# Verificação dos resultados do artigo (EDEPA 2026)

Este notebook roda **no navegador**, sem instalar nada no seu computador. Ele refaz as duas verificações do artigo:

1. **Parte 1:** resolve o modelo de Programação Linear pelo Método Simplex e confere as Tabelas 3, 4 e 5.
2. **Parte 2:** roda de novo a ferramenta Bandit nos 15 códigos gerados pelas IAs e mostra os alertas da Tabela 2.

**Como usar:** clique em cada célula cinza, em ordem, e aperte o botão ▶ à esquerda (ou `Shift + Enter`). Cada uma leva poucos segundos. Se o Colab avisar que o notebook não foi criado pelo Google, clique em "Executar mesmo assim".

O notebook também funciona fora do Colab (por exemplo, no VS Code ou no Jupyter), aberto de dentro da pasta do repositório. Os resultados são gravados na pasta `resultado_notebook/`.

## Preparação: localizar ou baixar os arquivos do trabalho

In [ ]:
import os, subprocess, sys

if os.path.exists('modelo_simplex/simplex_edepa.py'):
    pass                                    # já estamos dentro do repositório (execução local)
elif os.path.exists('edepa-devsecops-simplex'):
    os.chdir('edepa-devsecops-simplex')     # repositório já baixado nesta sessão
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/germanoafb/edepa-devsecops-simplex.git'], check=True)
    os.chdir('edepa-devsecops-simplex')

SAIDA = os.path.abspath('resultado_notebook')
print('Pasta do trabalho:', os.getcwd())

## Parte 1: Modelo de Programação Linear

Maximizar **Z = 4x₁ + 6x₂**, sujeito a x₁ + x₂ ≤ 40, x₁ ≤ 24, x₂ ≤ 20 e x₁, x₂ ≥ 0.

O script resolve o modelo por três métodos independentes (Simplex tabular com frações exatas, enumeração de todos os vértices da região viável e o solver HiGHS) e confere automaticamente cada número do artigo. A última linha deve dizer **"todas as verificações passaram"**.

In [ ]:
!"{sys.executable}" modelo_simplex/simplex_edepa.py --saida "{SAIDA}/modelo"

A célula abaixo mostra o mesmo resultado formatado em tabelas: quadros do Simplex, cenários de sensibilidade e lista de verificações.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{SAIDA}/modelo/relatorio.md', encoding='utf-8').read()))

## Parte 2: Ensaio com a ferramenta Bandit

O **Bandit** é um programa que lê código Python e aponta trechos potencialmente inseguros, os **alertas**. Cada alerta tem uma **severidade** (gravidade: baixa, média ou alta) e uma **confiança** (o quanto a ferramenta tem certeza).

Um alerta **não é** uma vulnerabilidade confirmada: confirmar exige a leitura humana do código. O artigo separa as duas coisas.

A célula abaixo instala o Bandit na versão usada no artigo (1.9.4) e o roda nos 15 códigos (3 IAs × 5 tarefas).

In [ ]:
%pip install -q bandit==1.9.4 pandas
!"{sys.executable}" bandit_reproducao/executar_bandit.py bandit_reproducao/codigos_originais --rotulo reexecucao --saida "{SAIDA}/bandit"

**Alertas encontrados pelo Bandit.** Leia `chatgpt/t4.py` como "código do ChatGPT para a tarefa 4". O arquivo `claude/comum.py` contém a parte compartilhada do código do Claude (o `import subprocess` usado na tarefa 4). Por isso o artigo conta **5 de 15 tarefas** sinalizadas, enquanto o script, que conta arquivos, mostra 6 de 16.

In [ ]:
import pandas as pd
alertas = pd.read_csv(f'{SAIDA}/bandit/reexecucao/alertas.csv')
alertas[['arquivo', 'linha', 'teste_id', 'severidade', 'confianca', 'cwe_scanner']]

**Classificação humana de cada alerta e achados da leitura manual**, conferida pelo autor. Ela vem do repositório: a reexecução acima só refaz a parte automática.

- `falso_positivo`: a leitura do código mostrou que o alerta não corresponde a um problema real;
- `nao_confirmada`: fraqueza plausível, mas sem exploração verificada;
- `sem_achado`: nada encontrado na leitura manual.

In [ ]:
classificacao = pd.read_csv('bandit_reproducao/saida/experimento/classificacao_manual.csv')
classificacao.columns = ['arquivo', 'origem', 'teste_ou_achado', 'linha', 'cwe_scanner', 'cwe_humano',
                         'classificacao', 'justificativa', 'evidencia', 'mitigacao_proposta', 'revisor', 'data']
pd.set_option('display.max_colwidth', 120)
classificacao[['arquivo', 'origem', 'teste_ou_achado', 'classificacao', 'justificativa']]

## Resumo

| Parte | O que foi verificado | O que **não** foi verificado |
|---|---|---|
| Modelo de PL | A solução (20, 20) com Z = 200, as folgas, os quadros do Simplex e os três cenários de sensibilidade estão matematicamente corretos | Se os coeficientes 4 e 6 e os limites 40, 24 e 20 correspondem à realidade (são hipóteses declaradas) |
| Ensaio com Bandit | Os alertas da Tabela 2 são reproduzidos pela ferramenta | Novas gerações das IAs dariam códigos diferentes; o ensaio é exploratório (15 trechos) |